In [ ]:
import numpy as np

class Layer():
    def __init__(self,):
        pass
    def forward(self, x):
        pass
    def __call__(self, x):
        return self.forward(x)


class NeuralNetwork:
    def __init__(self, layers):
        self.layers = layers
        
    def __call__(self, x):
        return self.forward(x)
    
    def forward(self, x):
        output = x
        for layer in self.layers:
            output = layer(output)
        return output
    
    def num_params(self):
        num = 0
        for layer in self.layers:
            num += layer.num_params()
        return num
        

class LinearLayer():
    def __init__(self, input_size, output_size, W=None, b=None):
        if W is None:
            W = np.random.randn(output_size, input_size) * 0.01
        if b is None:
            b = np.zeros(output_size)
        self.W = W
        self.b = b

    def forward(self, x):
        return np.dot(x, self.W.T) + self.b

    def __call__(self, x):
        return self.forward(x)

    def num_params(self):
        return 0
        
        
class ReLU():
    def forward(self, x):
        return np.maximum(0, x)

    def __call__(self, x):
        return self.forward(x)

    def num_params(self):
        return 0
    
class Sigmoid():
    def forward(self, x):
        return 1 / (1 + np.exp(-x))

    def __call__(self, x):
        return self.forward(x)

    def num_params(self):
        return 0
    
class Softmax():
    def forward(self, x):
        exp_shifted = np.exp(x - np.max(x, axis=-1, keepdims=True))
        return exp_shifted / np.sum(exp_shifted, axis=-1, keepdims=True)

    def __call__(self, x):
        return self.forward(x)

    def num_params(self):
        return 0
    

class SimpleConv():
    """
    Простейшая 2D свёртка без padding, stride=1.
    x.shape = (H, W)
    kernel.shape = (kH, kW)
    """
    def __init__(self, kernel):
        self.kernel = kernel

    def forward(self, x):
        H, W = x.shape
        kH, kW = self.kernel.shape
        out_H = H - kH + 1
        out_W = W - kW + 1
        output = np.zeros((out_H, out_W))

        for i in range(out_H):
            for j in range(out_W):
                region = x[i:i+kH, j:j+kW]
                output[i, j] = np.sum(region * self.kernel)
        return output

    def __call__(self, x):
        return self.forward(x)

    def num_params(self):
        return self.kernel.size
    
class Conv():
    def __init__(self, kernel_size, padding, stride, input_channels, output_channels):
        if isinstance(kernel_size, int):
            self.kernel_size = (kernel_size, kernel_size)
        else:
            self.kernel_size = kernel_size
        if isinstance(padding, int):
            self.padding = (padding, padding)
        else:
            self.padding = padding
        if isinstance(stride, int):
            self.stride = (stride, stride)
        else:
            self.stride = stride
        self.input_channels = input_channels
        self.output_channels = output_channels
        
        kH, kW = self.kernel_size
        limit = 1 / np.sqrt(input_channels * kH * kW)
        self.W = np.random.uniform(-limit, limit, (output_channels, input_channels, kH, kW))
        self.b = np.random.uniform(-limit, limit, output_channels)

    def forward(self, x):
        batch_size, in_channels, H, W = x.shape
        kH, kW = self.kernel_size
        padH, padW = self.padding
        strideH, strideW = self.stride
        
        H_padded = H + 2 * padH
        W_padded = W + 2 * padW
        
        # Pad input
        x_padded = np.pad(x, ((0,0), (0,0), (padH, padH), (padW, padW)), mode='constant')
        
        # Calculate output dimensions
        out_H = (H_padded - kH) // strideH + 1
        out_W = (W_padded - kW) // strideW + 1
        
        output = np.zeros((batch_size, self.output_channels, out_H, out_W))
        
        for n in range(batch_size):
            for oc in range(self.output_channels):
                for i in range(out_H):
                    for j in range(out_W):
                        h_start = i * strideH
                        w_start = j * strideW
                        h_end = h_start + kH
                        w_end = w_start + kW
                        region = x_padded[n, :, h_start:h_end, w_start:w_end]
                        output[n, oc, i, j] = np.sum(region * self.W[oc]) + self.b[oc]
        return output
    
    def __call__(self, x):
        return self.forward(x)

    def num_params(self):
        return self.W.size + self.b.size
        

In [15]:
neural_network = NeuralNetwork(layers=[LinearLayer(784, 392),
                                       LinearLayer(392, 196),
                                       LinearLayer(196, 98),
                                       LinearLayer(98, 49),
                                       LinearLayer(49, 10),])

neural_network.num_params()

0

In [16]:
x = np.random.rand(784)
y = neural_network(x)

print(y.shape)

(10,)


Задание:
1. Реализовать классы для Softmax, ReLU = max(0,x) и Sigmoid
2. Реализовать класс SimpleConv


In [17]:
x = np.random.rand(32, 784)
y = neural_network(x)